<a href="https://colab.research.google.com/github/Mohammmedsaqib/Neural-Networks-And-Fuzzy-Logic/blob/main/Assignment_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task
Develop and train a three-layer neural network on the Iris dataset ('Iris.csv') using ReLU activation for the hidden layer and Softmax for the output layer. The network should have 4 input neurons, 8 hidden neurons, and 3 output neurons. Implement forward and backward propagation, update parameters using gradient descent (learning rate 0.01) for 500 epochs, and evaluate the model's accuracy, precision, recall, and F1-score on a test set. Finally, summarize the network's architecture, training process, and performance metrics.

## Load and Preprocess Data

### Subtask:
Load the Iris dataset from 'Iris.csv'. Normalize feature values to the range [0, 1]. Convert target labels to one-hot encoding. Split the dataset into training and testing sets.


**Reasoning**:
First, I will load the 'Iris.csv' dataset, inspect its structure, then separate features and target, normalize the features, one-hot encode the target, and finally split the data into training and testing sets, all within a single code block.



In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

# 1. Load the 'Iris.csv' dataset
df = pd.read_csv('/content/Iris.csv')
print("First 5 rows of the dataset:")
print(df.head())
print("\nColumn names:")
print(df.columns)

# 2. Separate features (X) and target (y)
X = df.drop(['Id', 'Species'], axis=1)  # Assuming 'Id' is not a feature and 'Species' is the target
y = df['Species']

# 3. Normalize feature columns to a [0, 1] range
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)
print("\nFirst 5 rows of normalized features:")
print(X_scaled_df.head())

# 4. Convert the target variable into one-hot encoded format
encoder = OneHotEncoder(sparse_output=False) # sparse_output=False for dense array output
y_encoded = encoder.fit_transform(y.values.reshape(-1, 1))
y_encoded_df = pd.DataFrame(y_encoded, columns=encoder.get_feature_names_out(['Species']))
print("\nFirst 5 rows of one-hot encoded target:")
print(y_encoded_df.head())

# 5. Split the dataset into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X_scaled_df, y_encoded_df, test_size=0.2, random_state=42)

print(f"\nShape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

First 5 rows of the dataset:
   Id  SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm      Species
0   1            5.1           3.5            1.4           0.2  Iris-setosa
1   2            4.9           3.0            1.4           0.2  Iris-setosa
2   3            4.7           3.2            1.3           0.2  Iris-setosa
3   4            4.6           3.1            1.5           0.2  Iris-setosa
4   5            5.0           3.6            1.4           0.2  Iris-setosa

Column names:
Index(['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm',
       'Species'],
      dtype='object')

First 5 rows of normalized features:
   SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm
0       0.222222      0.625000       0.067797      0.041667
1       0.166667      0.416667       0.067797      0.041667
2       0.111111      0.500000       0.050847      0.041667
3       0.083333      0.458333       0.084746      0.041667
4       0.194444      0.666667      

## Initialize Network Parameters

### Subtask:
Initialize the weights and biases for a three-layer neural network: input (4 neurons), hidden (8 neurons), and output (3 neurons).


**Reasoning**:
To initialize the network parameters as requested, I will import numpy, define the neuron counts for each layer, and then initialize the weight matrices (W1, W2) with small random values and bias vectors (b1, b2) with zeros.



In [2]:
import numpy as np

# 2. Define the number of input, hidden, and output neurons
n_input = 4
n_hidden = 8
n_output = 3

print(f"Network architecture: {n_input} input neurons, {n_hidden} hidden neurons, {n_output} output neurons")

# 3. Initialize the weight matrix W1 for the hidden layer
W1 = np.random.randn(n_input, n_hidden) * 0.01
print(f"\nShape of W1 (input to hidden weights): {W1.shape}")

# 4. Initialize the bias vector b1 for the hidden layer
b1 = np.zeros((1, n_hidden))
print(f"Shape of b1 (hidden layer biases): {b1.shape}")

# 5. Initialize the weight matrix W2 for the output layer
W2 = np.random.randn(n_hidden, n_output) * 0.01
print(f"Shape of W2 (hidden to output weights): {W2.shape}")

# 6. Initialize the bias vector b2 for the output layer
b2 = np.zeros((1, n_output))
print(f"Shape of b2 (output layer biases): {b2.shape}")

print("\nNetwork parameters initialized successfully.")

Network architecture: 4 input neurons, 8 hidden neurons, 3 output neurons

Shape of W1 (input to hidden weights): (4, 8)
Shape of b1 (hidden layer biases): (1, 8)
Shape of W2 (hidden to output weights): (8, 3)
Shape of b2 (output layer biases): (1, 3)

Network parameters initialized successfully.


## Implement Activation Functions

### Subtask:
Implement the ReLU activation function and its derivative for the hidden layer, and the Softmax activation function for the output layer.


**Reasoning**:
To implement the requested activation functions, I will define `relu`, `relu_derivative`, and `softmax` functions using NumPy to ensure numerical stability and efficiency.



In [3]:
import numpy as np

def relu(x):
    """ReLU activation function"""
    return np.maximum(0, x)

def relu_derivative(x):
    """Derivative of the ReLU activation function"""
    return (x > 0).astype(float)

def softmax(x):
    """Softmax activation function"""
    # Subtract the maximum value for numerical stability
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

print("Activation functions (ReLU, ReLU derivative, Softmax) defined.")

Activation functions (ReLU, ReLU derivative, Softmax) defined.


## Implement Forward Propagation

### Subtask:
Develop the forward propagation algorithm to compute the output of the neural network given an input.


**Reasoning**:
To implement the forward propagation, I will define a function that takes inputs, weights, and biases, then perform the weighted sums and apply the ReLU and Softmax activation functions as specified.



In [4]:
def forward_propagation(X, W1, b1, W2, b2):
    """Performs forward propagation through the neural network."""

    # Convert X (DataFrame) to numpy array if it's not already, for consistent operations
    if isinstance(X, pd.DataFrame):
        X_np = X.to_numpy()
    else:
        X_np = X

    # Layer 1: Input to Hidden Layer
    Z1 = np.dot(X_np, W1) + b1
    A1 = relu(Z1) # Apply ReLU activation

    # Layer 2: Hidden to Output Layer
    Z2 = np.dot(A1, W2) + b2
    A2 = softmax(Z2) # Apply Softmax activation

    return Z1, A1, Z2, A2

print("Forward propagation function defined.")

Forward propagation function defined.


## Implement Backward Propagation

### Subtask:
Develop the backpropagation algorithm to compute the gradients of the loss function with respect to the weights and biases.


**Reasoning**:
To implement the backward propagation algorithm, I will define a function that calculates the gradients of the loss with respect to the weights and biases for both the output and hidden layers, following the provided steps for error calculation and gradient computation.



In [5]:
def backward_propagation(X, y_true, Z1, A1, A2, W1, W2):
    """Performs backward propagation to compute gradients."""

    m = X.shape[0] # Number of samples

    # Ensure X is a numpy array
    if isinstance(X, pd.DataFrame):
        X_np = X.to_numpy()
    else:
        X_np = X

    # Layer 2: Output Layer Gradients
    dZ2 = A2 - y_true # Gradient of loss with respect to Z2
    dW2 = (1/m) * np.dot(A1.T, dZ2) # Gradient of loss with respect to W2
    db2 = (1/m) * np.sum(dZ2, axis=0, keepdims=True) # Gradient of loss with respect to b2

    # Layer 1: Hidden Layer Gradients
    dA1 = np.dot(dZ2, W2.T) # Gradient of loss with respect to A1
    dZ1 = dA1 * relu_derivative(Z1) # Gradient of loss with respect to Z1 (applying ReLU derivative)
    dW1 = (1/m) * np.dot(X_np.T, dZ1) # Gradient of loss with respect to W1
    db1 = (1/m) * np.sum(dZ1, axis=0, keepdims=True) # Gradient of loss with respect to b1

    return dW1, db1, dW2, db2

print("Backward propagation function defined.")

Backward propagation function defined.


## Implement Parameter Update

### Subtask:
Implement the function to update the network's weights and biases using gradient descent with a learning rate of 0.01.


**Reasoning**:
To implement the parameter update, I will define a function that takes the current weights and biases, their calculated gradients, and the learning rate, then apply the gradient descent update rule to modify each parameter, and finally return the updated parameters.

**Reasoning**:
To implement the parameter update function, I will define a Python function `update_parameters` that applies the gradient descent rule to the weights and biases using a specified learning rate.



In [6]:
def update_parameters(W1, b1, W2, b2, dW1, db1, dW2, db2, learning_rate):
    """Updates the network's weights and biases using gradient descent."""

    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1
    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2

    return W1, b1, W2, b2

# Define the learning rate
learning_rate = 0.01

print(f"Parameter update function defined with a learning rate of {learning_rate}.")

Parameter update function defined with a learning rate of 0.01.


## Train the Neural Network

### Subtask:
Train the neural network for 500 epochs, iteratively performing forward propagation, backward propagation, and parameter updates. Keep track of the loss during training.


**Reasoning**:
To train the neural network, I will implement a training loop that performs forward propagation, calculates the loss, executes backward propagation to compute gradients, and updates the parameters using gradient descent for 500 epochs, storing the loss at each step.



In [7]:
epochs = 500
loss_history = []

# Convert y_train to numpy array once for efficiency, as it's used repeatedly
y_train_np = y_train.to_numpy()
m = X_train.shape[0] # Number of samples

print("Starting training...")
for epoch in range(epochs):
    # 1. Forward Propagation
    Z1, A1, Z2, A2 = forward_propagation(X_train, W1, b1, W2, b2)

    # 2. Calculate Loss (Cross-Entropy Loss)
    # Add a small epsilon to A2 to prevent log(0) issues
    loss = -np.sum(y_train_np * np.log(A2 + 1e-9)) / m
    loss_history.append(loss)

    # 3. Backward Propagation
    dW1, db1, dW2, db2 = backward_propagation(X_train, y_train_np, Z1, A1, A2, W1, W2)

    # 4. Update Parameters
    W1, b1, W2, b2 = update_parameters(W1, b1, W2, b2, dW1, db1, dW2, db2, learning_rate)

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss:.4f}")

print("Training complete.")
print(f"Final Loss: {loss_history[-1]:.4f}")

Starting training...
Epoch 100/500, Loss: 1.0984
Epoch 200/500, Loss: 1.0982
Epoch 300/500, Loss: 1.0979
Epoch 400/500, Loss: 1.0976
Epoch 500/500, Loss: 1.0970
Training complete.
Final Loss: 1.0970


## Evaluate Model Performance

### Subtask:
Evaluate the trained model on the test set. Calculate and display the accuracy, precision, recall, and F1-score.


**Reasoning**:
To evaluate the model, I will use the trained weights and biases to perform forward propagation on the test data, convert the one-hot encoded true labels and predicted probabilities into single class labels, and then calculate and display accuracy, precision, recall, and F1-score using scikit-learn's metrics functions.



In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Perform forward propagation on the test data
Z1_test, A1_test, Z2_test, A2_test = forward_propagation(X_test, W1, b1, W2, b2)

# 2. Convert one-hot encoded true labels (y_test) to single class labels
y_true_labels = np.argmax(y_test.to_numpy(), axis=1)

# 3. Convert predicted probabilities (A2_test) to predicted class labels
y_pred_labels = np.argmax(A2_test, axis=1)

# 4. Calculate evaluation metrics
accuracy = accuracy_score(y_true_labels, y_pred_labels)
precision = precision_score(y_true_labels, y_pred_labels, average='weighted', zero_division=0)
recall = recall_score(y_true_labels, y_pred_labels, average='weighted', zero_division=0)
f1 = f1_score(y_true_labels, y_pred_labels, average='weighted', zero_division=0)

# 5. Print the evaluation metrics
print("\nModel Performance on Test Set:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")


Model Performance on Test Set:
Accuracy: 0.3000
Precision: 0.0900
Recall: 0.3000
F1-Score: 0.1385


## Final Task

### Subtask:
Summarize the implementation of the neural network, including the architecture, training process, and the performance metrics obtained on the test set.


## Summary:

### Q&A

1.  **Summarize the implementation of the neural network, including the architecture, training process, and the performance metrics obtained on the test set.**
    *   **Architecture**: The neural network is a three-layer feedforward network with 4 input neurons, 8 hidden neurons using ReLU activation, and 3 output neurons using Softmax activation.
    *   **Training Process**: The network was trained for 500 epochs using gradient descent with a learning rate of 0.01. During each epoch, forward propagation computed outputs, cross-entropy loss was calculated, and then backward propagation computed gradients, which were used to update weights and biases. The training loss decreased from 1.0984 at epoch 100 to 1.0970 at epoch 500.
    *   **Performance Metrics on Test Set**:
        *   Accuracy: 0.3000
        *   Precision: 0.0900
        *   Recall: 0.3000
        *   F1-Score: 0.1385

### Data Analysis Key Findings

*   **Data Preprocessing**: The Iris dataset was successfully loaded. Features were normalized to a \[0, 1] range, and target labels were one-hot encoded. The dataset was split into training (120 samples) and testing (30 samples) sets.
*   **Network Architecture and Initialization**: The neural network was set up with 4 input, 8 hidden, and 3 output neurons. Weights were initialized with small random values (`* 0.01`), and biases were initialized to zeros, with correct shapes (e.g., `W1`: (4, 8), `b1`: (1, 8), `W2`: (8, 3), `b2`: (1, 3)).
*   **Activation Functions**: ReLU and its derivative, along with the numerically stable Softmax function, were correctly implemented.
*   **Training Dynamics**: Over 500 epochs, the training loss decreased from 1.0984 (at epoch 100) to 1.0970 (at epoch 500), indicating that the model was learning, albeit slowly.
*   **Model Performance**: The model achieved very low performance on the test set:
    *   Accuracy: 30.0%
    *   Precision: 9.0%
    *   Recall: 30.0%
    *   F1-Score: 13.85%

### Insights or Next Steps

*   **Suboptimal Performance**: The current performance metrics (e.g., 30% accuracy) indicate that the neural network is not effectively classifying the Iris dataset. This suggests issues with the model, training process, or hyperparameters.
*   **Hyperparameter Tuning and Regularization**: To improve performance, consider increasing the number of epochs, adjusting the learning rate (potentially trying a larger value or a learning rate schedule), using a more sophisticated weight initialization method (e.g., Kaiming or Xavier initialization), and/or adding regularization techniques (e.g., L1/L2 regularization, dropout) to prevent overfitting or improve convergence. A larger hidden layer or deeper network could also be explored if the current capacity is insufficient.
